# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
import pandas as pd

url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

# Check the columns before combining
for name, df in [("file1", df1), ("file2", df2), ("file3", df3)]:
    print(name, df.shape)
    print(list(df.columns), "\n")

file1 (4008, 11)
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount'] 

file2 (996, 11)
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class'] 

file3 (7070, 11)
['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class'] 



In [3]:
# Standardize column names before concatenating

def standardize_columns(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    df = df.rename(columns={"st": "state"})
    return df

df1 = standardize_columns(df1)
df2 = standardize_columns(df2)
df3 = standardize_columns(df3)

# All three should now have the same set of columns
print(set(df1.columns) == set(df2.columns) == set(df3.columns))

True


In [4]:
# combine data 

combined = pd.concat([df1, df2, df3], axis=0, ignore_index=True)
print(combined.shape)
combined.head()

(12074, 11)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [7]:
# cleaning the combine data for the functions

def clean_open_complaints(value):
    """Turns '1/5/00' into 5. Leaves plain numbers as they are."""
    if isinstance(value, str) and "/" in value:
        return value.split("/")[1]
    return value

def clean_data(df):
    df = df.copy()

    # 1. Drop rows that are completely empty
    df = df.dropna(how="all")

    # 2. Gender: keep only "M" and "F"
    df["gender"] = df["gender"].replace({
        "Femal": "F", "female": "F", "Female": "F",
        "Male": "M", "male": "M"
    })

    # 3. State: use full names
    df["state"] = df["state"].replace({
        "AZ": "Arizona", "Cali": "California", "WA": "Washington"
    })

    # 4. Education
    df["education"] = df["education"].replace({"Bachelors": "Bachelor"})

    # 5. Vehicle class: group the luxury categories
    df["vehicle_class"] = df["vehicle_class"].replace({
        "Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"
    })

    # 6. Customer lifetime value: remove "%" and convert to a number
    df["customer_lifetime_value"] = pd.to_numeric(
        df["customer_lifetime_value"].astype(str).str.replace("%", "", regex=False),
        errors="coerce"
    )

    # 7. Open complaints: take the middle value, then convert to a number
    df["number_of_open_complaints"] = pd.to_numeric(
        df["number_of_open_complaints"].apply(clean_open_complaints),
        errors="coerce"
    )

    # 8. Missing values: median for numbers, most frequent value for text
    for col in df.select_dtypes(include="number").columns:
        df[col] = df[col].fillna(df[col].median())
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna(df[col].mode()[0])

    # 9. Numeric columns as integers
    for col in df.select_dtypes(include="number").columns:
        df[col] = df[col].astype(int)

    # 10. Remove duplicates and reset the index
    df = df.drop_duplicates().reset_index(drop=True)

    return df

df

,Customer,State,Customer Lifetime Value,Education,Gender,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Total Claim Amount,Vehicle Class
0,SA25987,Washington,3479.137523,High School or Below,M,0,104,0,Personal Auto,499.200000,Two-Door Car
1,TB86706,Arizona,2502.637401,Master,M,0,66,0,Personal Auto,3.468912,Two-Door Car
2,ZL73902,Nevada,3265.156348,Bachelor,F,25820,82,0,Personal Auto,393.600000,Four-Door Car
3,KX23516,California,4455.843406,High School or Below,F,0,121,0,Personal Auto,699.615192,SUV
4,FN77294,California,7704.958480,High School or Below,M,30366,101,2,Personal Auto,484.800000,SUV
...,...,...,...,...,...,...,...,...,...,...,...
7065,LA72316,California,23405.987980,Bachelor,M,71941,73,0,Personal Auto,198.234764,Four-Door Car
7066,PK87824,California,3096.511217,College,F,21604,79,0,Corporate Auto,379.200000,Four-Door Car
7067,TD14365,California,8163.890428,Bachelor,M,0,85,3,Corporate Auto,790.784983,Four-Door Car
7068,UP19263,California,7524.442436,College,M,21941,96,0,Personal Auto,691.200000,Four-Door Car


In [8]:
# let's the function now and see if it works

clean_df = clean_data(combined)

print(clean_df.shape)
print(clean_df.isna().sum())      # should all be 0
print(clean_df.dtypes)

# The categories should now be consistent
for col in ["gender", "state", "education", "vehicle_class"]:
    print(col, clean_df[col].unique())

# Optional: save the result for the next challenges
clean_df.to_csv("clean_combined_data.csv", index=False)

(9134, 11)
customer                     0
state                        0
gender                       0
education                    0
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  0
vehicle_class                0
total_claim_amount           0
dtype: int64
customer                       str
state                          str
gender                         str
education                      str
customer_lifetime_value      int64
income                       int64
monthly_premium_auto         int64
number_of_open_complaints    int64
policy_type                    str
vehicle_class                  str
total_claim_amount           int64
dtype: object
gender <ArrowStringArray>
['F', 'M']
Length: 2, dtype: str
state <ArrowStringArray>
['Washington', 'Arizona', 'Nevada', 'California', 'Oregon']
Length: 5, dtype: str
education <ArrowStringArray>
['Master', 'Bachelor', 'High School or Be

/var/folders/07/xlg6wrm96db55dkn2bnl5vnw0000gn/T/ipykernel_24108/1023462824.py:49: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)

# The first column is an old index saved by mistake, so drop it
df = df.drop(columns=["unnamed:_0"])

print(df.shape)
print(df.dtypes)
df.head()

(10910, 26)
customer                             str
state                                str
customer_lifetime_value          float64
response                             str
coverage                             str
education                            str
effective_to_date                    str
employmentstatus                     str
gender                               str
income                             int64
location_code                        str
marital_status                       str
monthly_premium_auto               int64
months_since_last_claim          float64
months_since_policy_inception      int64
number_of_open_complaints        float64
number_of_policies                 int64
policy_type                          str
policy                               str
renew_offer_type                     str
sales_channel                        str
total_claim_amount               float64
vehicle_class                        str
vehicle_size                         str
vehi

,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,48029,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,0,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,22139,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,49078,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,23675,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


In [10]:
# revenue per sales chanel 

revenue_by_channel = df.pivot_table(
    index="sales_channel",
    values="total_claim_amount",
    aggfunc="sum"
).round(2)

revenue_by_channel = revenue_by_channel.sort_values(
    by="total_claim_amount", ascending=False
)
revenue_by_channel

,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


In [11]:
df.pivot_table(
    index="sales_channel",
    values="total_claim_amount",
    aggfunc=["sum", "count", "mean"]
).round(2)

,sum,count,mean
,total_claim_amount,total_claim_amount,total_claim_amount
sales_channel,,,
Agent,1810226.82,4121,439.27
Branch,1301204.00,3022,430.58
Call Center,926600.82,2141,432.79
Web,706600.04,1626,434.56


In [12]:
# Average customer lifetime value by gender and education

clv_pivot = df.pivot_table(
    index="education",
    columns="gender",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

clv_pivot

gender,F,M
education,,
Bachelor,7874.27,7703.60
College,7748.82,8052.46
Doctor,7328.51,7415.33
High School or Below,8675.22,8149.69
Master,8157.05,8168.83


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here